# Experiment 1 - XOR Gate using a Multilayer Perceptron

**Aim:** Implement the XOR logic gate with a multilayer perceptron (MLP) and
verify that a network with a nonlinear hidden layer can learn a problem that
is not linearly separable.

**Theory in brief:** A single sigmoid neuron can only separate data with one
straight line, so it cannot represent XOR. An MLP with a hidden layer learns an
internal nonlinear representation. This notebook uses the architecture from
the lab sheet: 2 input neurons, 4 hidden neurons with `tanh`, and 1 output
neuron with `sigmoid`. It is trained with Adam and binary cross-entropy.

**Dataset:** The four-row XOR truth table is defined directly in code. No
download and no local dataset file are needed.

**Expected result (original lab record):** 100% accuracy on all four XOR rows
after 1000 epochs.

**Runtime:** Under 10 seconds on CPU.

> This notebook is self-contained. Run all cells from top to bottom.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Data: XOR truth table

The input matrix `X` has one row per truth-table entry. The target `y` is 1
when exactly one input is 1.

In [ ]:
X = np.array(
    [
        [0.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ],
    dtype=np.float32,
)
y = np.array([[0.0], [1.0], [1.0], [0.0]], dtype=np.float32)

assert X.shape == (4, 2)
assert y.shape == (4, 1)

print(" A  B | XOR")
print("-------+----")
for a, b, target in zip(X[:, 0], X[:, 1], y[:, 0]):
    print(f" {int(a)}  {int(b)} |  {int(target)}")

## 3. Model: 2-4-1 MLP

- **Hidden layer:** 4 units, `tanh` activation. This is the nonlinear layer
  that makes XOR learnable.
- **Output layer:** 1 unit, `sigmoid` activation, interpreted as
  `P(target = 1)`.
- **Loss:** binary cross-entropy.
- **Optimizer:** Adam with learning rate 0.05, matching the lab sheet.

In [ ]:
def build_xor_model(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(2,), name="xor_input"),
            tf.keras.layers.Dense(4, activation="tanh", name="hidden"),
            tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
        ],
        name="xor_mlp",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_xor_model()
model.summary()

## 4. Training

The network is small enough to train for the full 1000 epochs specified in the
lab sheet.

In [ ]:
start_time = time.perf_counter()
history = model.fit(
    X,
    y,
    epochs=1000,
    batch_size=4,
    shuffle=True,
    verbose=0,
)
training_seconds = time.perf_counter() - start_time

print(f"Training finished in {training_seconds:.2f} seconds")
print(f"Final training loss    : {history.history['loss'][-1]:.6f}")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")

## 5. Evaluation on the complete truth table

In [ ]:
test_loss, test_accuracy = model.evaluate(X, y, verbose=0)
probabilities = model.predict(X, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
targets = y.ravel().astype(int)

print(f"Test loss    : {test_loss:.6f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print()
print(" A  B | Expected | P(1)     | Predicted | Correct")
print("-------+----------+----------+-----------+--------")
for a, b, target, probability, prediction in zip(
    X[:, 0], X[:, 1], targets, probabilities, predictions
):
    print(
        f" {int(a)}  {int(b)} |    {target}     | {probability:.6f} |"
        f"     {prediction}     |   {prediction == target}"
    )

assert test_accuracy == 1.0, "The MLP did not fit the XOR truth table."

## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(history.history["loss"], color="tab:blue")
axes[0].set_title("Binary cross-entropy loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["accuracy"], color="tab:green")
axes[1].set_title("Training accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp01_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Inference

`predict_xor` accepts any pair of binary inputs and returns the predicted XOR
value together with the sigmoid probability.

In [ ]:
def predict_xor(a, b, threshold=0.5):
    sample = np.array([[a, b]], dtype=np.float32)
    probability = float(model.predict(sample, verbose=0)[0, 0])
    prediction = int(probability >= threshold)
    return prediction, probability


for a, b in [(0, 0), (0, 1), (1, 0), (1, 1), (2, -1)]:
    prediction, probability = predict_xor(a, b)
    print(f"XOR({a}, {b}) = {prediction}  (P(1) = {probability:.6f})")

The last test `(2, -1)` is outside the training distribution. It is shown only
to demonstrate the inference API; the network has not been trained to treat
those inputs as valid logic levels.

## 8. Save and reload the trained model

In [ ]:
model_path = OUTPUT_DIR / "exp01_xor_mlp.keras"
model.save(model_path)
print(f"Saved model to: {model_path.resolve()}")

reloaded_model = tf.keras.models.load_model(model_path)
reloaded_probabilities = reloaded_model.predict(X, verbose=0).ravel()
reloaded_predictions = (reloaded_probabilities >= 0.5).astype(int)

print("Reloaded predictions:", reloaded_predictions.tolist())
assert np.array_equal(reloaded_predictions, predictions)
print("Reload check passed: predictions are identical.")

## 9. Conclusion

The 2-4-1 MLP learned all four XOR mappings and reached 100% accuracy on the
truth table. This confirms the central lesson of the experiment: a hidden
nonlinear layer turns a model that cannot separate XOR into one that can.

**Artifacts created at runtime (not committed):**

- `outputs/exp01_xor_mlp.keras`
- `outputs/exp01_training_curves.png`

**What to report:** training loss/accuracy, final truth-table predictions, and
the fact that the model solved XOR perfectly. The original lab sheet records
100% accuracy, which matches the assertion in this notebook.

**Note on execution:** the notebook was syntax-checked during repository
preparation, but the full training run was not performed here. Rerunning it
reproduces the result because the seed is fixed.